## IMPORT LINBRARIES

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## IMPORT DATASET

In [2]:
DATA_DIR = Path("data")

customers = pd.read_csv(DATA_DIR / "customers.csv")
subscriptions = pd.read_csv(DATA_DIR / "subscriptions.csv")
payments = pd.read_csv(DATA_DIR / "payments.csv")
marketing = pd.read_csv(DATA_DIR / "marketing_spend.csv")
usage = pd.read_csv(DATA_DIR / "usage_engagement.csv")
plan_changes = pd.read_csv(DATA_DIR / "plan_changes.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
datasets = {
    "Customers": customers,
    "Subscriptions": subscriptions,
    "Payments": payments,
    "Marketing": marketing,
    "Usage": usage,
    "Plan Changes": plan_changes
}

for name, df in datasets.items():
    print(f"{name:15} | Rows: {len(df):>8,} | Columns: {len(df.columns)}")

Customers       | Rows:   10,000 | Columns: 5
Subscriptions   | Rows:   10,113 | Columns: 8
Payments        | Rows:  109,303 | Columns: 5
Marketing       | Rows:      180 | Columns: 3
Usage           | Rows:  146,190 | Columns: 5
Plan Changes    | Rows:    2,887 | Columns: 5


In [4]:
display(customers.head())
display(subscriptions.head())
display(payments.head())

customers.info()
subscriptions.info()
payments.info()

,customer_id,signup_date,country,company_size,acquisition_channel
0,CUST002394,11/1/2023,Brazil,51-200,Paid Search
1,CUST000744,11/1/2023,India,51-200,Referral
2,CUST007122,11/1/2023,India,Nov-50,Social
3,CUST000766,11/1/2023,United States,10-Jan,Organic
4,CUST006737,11/1/2023,India,1000+,Paid Search


,subscription_id,customer_id,plan,billing_cycle,price,start_date,end_date,status
0,SUB0000001,CUST002394,Pro,annual,790.0,2023-11-01,2024-11-01,cancelled
1,SUB0000002,CUST000744,Pro,monthly,79.0,2023-11-01,2026-10-01,cancelled
2,SUB0000003,CUST007122,Pro,annual,790.0,2023-11-01,2026-10-01,cancelled
3,SUB0000004,CUST000766,Enterprise,monthly,249.0,2023-11-01,NaN,active
4,SUB0000005,CUST006737,Pro,monthly,79.0,2023-11-01,NaN,active


,payment_id,subscription_id,amount,payment_date,payment_status
0,PAY00000001,SUB0000001,790.0,2023-11-01,success
1,PAY00000002,SUB0000001,790.0,2024-11-01,success
2,PAY00000003,SUB0000002,79.0,2023-11-01,success
3,PAY00000004,SUB0000002,79.0,2023-12-01,success
4,PAY00000005,SUB0000002,79.0,2024-01-01,success


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          10000 non-null  object
 1   signup_date          10000 non-null  object
 2   country              10000 non-null  object
 3   company_size         10000 non-null  object
 4   acquisition_channel  10000 non-null  object
dtypes: object(5)
memory usage: 390.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10113 entries, 0 to 10112
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   subscription_id  10113 non-null  object 
 1   customer_id      10113 non-null  object 
 2   plan             10113 non-null  object 
 3   billing_cycle    10113 non-null  object 
 4   price            10113 non-null  float64
 5   start_date       10113 non-null  object 
 6   end_date         3406 non-n

In [5]:
for name, df in datasets.items():
    print(f"\n{name}")
    missing = df.isna().sum()
    print(missing[missing > 0].sort_values(ascending=False))


Customers
Series([], dtype: int64)

Subscriptions
end_date    6707
dtype: int64

Payments
Series([], dtype: int64)

Marketing
Series([], dtype: int64)

Usage
Series([], dtype: int64)

Plan Changes
Series([], dtype: int64)


In [6]:
primary_keys = {
    "Customers": (customers, ["customer_id"]),
    "Subscriptions": (subscriptions, ["subscription_id"]),
    "Payments": (payments, ["payment_id"]),
    "Marketing": (marketing, ["month", "channel"]),
    "Usage": (usage, ["customer_id", "month"]),
    "Plan Changes": (plan_changes, ["plan_change_id"])
}

for name, (df, keys) in primary_keys.items():
    print(name, "duplicate keys:", df.duplicated(keys).sum())

Customers duplicate keys: 0
Subscriptions duplicate keys: 0
Payments duplicate keys: 0
Marketing duplicate keys: 0
Usage duplicate keys: 0
Plan Changes duplicate keys: 0


In [7]:
def standardize_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    return df

customers = standardize_data(customers)
subscriptions = standardize_data(subscriptions)
payments = standardize_data(payments)
marketing = standardize_data(marketing)
usage = standardize_data(usage)
plan_changes = standardize_data(plan_changes)

In [8]:
date_columns = {
    "customers": ["signup_date"],
    "subscriptions": ["start_date", "end_date"],
    "payments": ["payment_date"],
    "plan_changes": ["change_date"]
}

frames = {
    "customers": customers,
    "subscriptions": subscriptions,
    "payments": payments,
    "plan_changes": plan_changes
}

for name, cols in date_columns.items():
    for col in cols:
        frames[name][col] = pd.to_datetime(
            frames[name][col], errors="coerce"
        )

subscriptions["price"] = pd.to_numeric(
    subscriptions["price"], errors="coerce"
)

payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)

marketing["spend"] = pd.to_numeric(
    marketing["spend"], errors="coerce"
)

for col in ["logins", "features_used", "support_tickets"]:
    usage[col] = pd.to_numeric(usage[col], errors="coerce")

In [9]:
assert customers["customer_id"].is_unique
assert subscriptions["subscription_id"].is_unique
assert payments["payment_id"].is_unique

assert subscriptions["price"].notna().all()
assert subscriptions["price"].ge(0).all()
assert payments["amount"].notna().all()
assert payments["amount"].ge(0).all()
assert marketing["spend"].notna().all()
assert marketing["spend"].ge(0).all()

assert set(subscriptions["status"].dropna().unique()) <= {
    "active", "cancelled"
}

assert set(payments["payment_status"].dropna().unique()) <= {
    "success", "failed", "refunded"
}

assert subscriptions.loc[
    subscriptions["status"] == "active", "end_date"
].isna().all()

assert subscriptions.loc[
    subscriptions["status"] == "cancelled", "end_date"
].notna().all()

assert subscriptions["customer_id"].isin(
    customers["customer_id"]
).all()

assert payments["subscription_id"].isin(
    subscriptions["subscription_id"]
).all()

print("All initial data-quality checks passed.")

All initial data-quality checks passed.


In [10]:
CLEAN_DIR = Path("data/cleaned")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

cleaned = {
    "customers": customers,
    "subscriptions": subscriptions,
    "payments": payments,
    "marketing_spend": marketing,
    "usage_engagement": usage,
    "plan_changes": plan_changes,
}

for name, df in cleaned.items():
    df.to_csv(CLEAN_DIR / f"{name}.csv", index=False)

print("Cleaned datasets saved.")

Cleaned datasets saved.


In [11]:
%pip install pandas sqlalchemy pymysql python-dotenv

In [12]:
env_content = """DB_HOST=localhost
DB_PORT=3306
DB_NAME=saas_analytics
DB_USER=saas_user
DB_PASSWORD=Ritesh_StrongPassword_123!"""

with open(".env", "w") as f:
    f.write(env_content)

print(".env file created successfully!")

.env file created successfully!


In [18]:
from sqlalchemy import text

with engine.connect() as conn:
    print("Connected database:", conn.execute(
        text("SELECT DATABASE()")
    ).scalar())

    result = conn.execute(text("""
        SELECT TABLE_NAME
        FROM information_schema.TABLES
        WHERE TABLE_SCHEMA = DATABASE()
        ORDER BY TABLE_NAME
    """))

    print("Tables visible to Python:", [row[0] for row in result])

Connected database: saas_analytics
Tables visible to Python: ['marketing_spend', 'payments', 'plan_changes', 'subscriptions', 'usage_engagement']


In [22]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    print([row[0] for row in result])

['customers', 'marketing_spend', 'payments', 'plan_changes', 'subscriptions', 'usage_engagement']


In [23]:
with engine.connect() as conn:
    for table in [
        "customers", "subscriptions", "payments",
        "marketing_spend", "usage_engagement", "plan_changes"
    ]:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()
        print(f"{table}: {count:,} rows")

customers: 0 rows
subscriptions: 10,113 rows
payments: 109,303 rows
marketing_spend: 180 rows
usage_engagement: 146,190 rows
plan_changes: 0 rows


In [25]:
from sqlalchemy import text

with engine.connect() as conn:
    for table in [
        "customers", "subscriptions", "payments",
        "marketing_spend", "usage_engagement", "plan_changes"
    ]:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()
        print(f"{table}: {count:,} rows")

customers: 10,000 rows
subscriptions: 10,113 rows
payments: 109,303 rows
marketing_spend: 180 rows
usage_engagement: 146,190 rows
plan_changes: 0 rows


In [26]:
import pandas as pd
from pathlib import Path

file_path = Path("data/cleaned/plan_changes.csv")
df = pd.read_csv(file_path)

print("Rows in CSV:", len(df))
print(df.head())
print(df.dtypes)

Rows in CSV: 2887
  plan_change_id subscription_id change_date    old_plan    new_plan
0     CHG0000001      SUB0000006  2026-02-09  Enterprise       Basic
1     CHG0000002      SUB0000009  2026-01-14       Basic         Pro
2     CHG0000003      SUB0000011  2026-06-02       Basic  Enterprise
3     CHG0000004      SUB0000013  2025-01-09       Basic         Pro
4     CHG0000005      SUB0000013  2026-05-26         Pro  Enterprise
plan_change_id     object
subscription_id    object
change_date        object
old_plan           object
new_plan           object
dtype: object


In [28]:
from pathlib import Path
import pandas as pd
from sqlalchemy import text

df = pd.read_csv(Path("data/cleaned/plan_changes.csv"))

print("CSV rows:", len(df))
print("CSV columns:", df.columns.tolist())
print("Sample records:")
display(df.head())

with engine.connect() as conn:
    print(
        "MySQL plan_changes rows:",
        conn.execute(
            text("SELECT COUNT(*) FROM plan_changes")
        ).scalar()
    )

CSV rows: 2887
CSV columns: ['plan_change_id', 'subscription_id', 'change_date', 'old_plan', 'new_plan']
Sample records:


,plan_change_id,subscription_id,change_date,old_plan,new_plan
0,CHG0000001,SUB0000006,2026-02-09,Enterprise,Basic
1,CHG0000002,SUB0000009,2026-01-14,Basic,Pro
2,CHG0000003,SUB0000011,2026-06-02,Basic,Enterprise
3,CHG0000004,SUB0000013,2025-01-09,Basic,Pro
4,CHG0000005,SUB0000013,2026-05-26,Pro,Enterprise


MySQL plan_changes rows: 0


In [29]:
import pandas as pd
from pathlib import Path
from sqlalchemy import text

# Load the cleaned CSV
df = pd.read_csv("data/cleaned/plan_changes.csv")

# Convert change_date to date format
df["change_date"] = pd.to_datetime(
    df["change_date"], errors="raise"
).dt.date

# Check that all subscription IDs exist in MySQL
with engine.connect() as conn:
    missing_ids = conn.execute(
        text("""
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT p.subscription_id
                FROM plan_changes p
                WHERE 1 = 0
            ) AS x
        """)
    ).scalar()

print("CSV records ready:", len(df))

CSV records ready: 2887


In [32]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(
        text("SHOW CREATE TABLE plan_changes")
    )
    print(result.fetchone()[1])

CREATE TABLE `plan_changes` (
  `subscription_id` varchar(20) NOT NULL,
  `change_date` date NOT NULL,
  `old_plan` varchar(30) DEFAULT NULL,
  `new_plan` varchar(30) DEFAULT NULL,
  PRIMARY KEY (`subscription_id`,`change_date`),
  CONSTRAINT `plan_changes_ibfk_1` FOREIGN KEY (`subscription_id`) REFERENCES `subscriptions` (`subscription_id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci


In [33]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE plan_changes
        ADD COLUMN plan_change_id VARCHAR(20) NULL
    """))

    conn.execute(text("""
        ALTER TABLE plan_changes
        ADD UNIQUE KEY uq_plan_change_id (plan_change_id)
    """))

print("Column added successfully.")

Column added successfully.


In [34]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM plan_changes")
    ).scalar()

print("Existing rows:", count)

Existing rows: 0


In [35]:
df.to_sql(
    name="plan_changes",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=500
)

print("Plan changes imported successfully!")

Plan changes imported successfully!


In [36]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM plan_changes")
    ).scalar()

print("Imported plan changes:", count)

Imported plan changes: 2887


In [37]:
from sqlalchemy import text

tables = [
    "customers",
    "subscriptions",
    "payments",
    "marketing_spend",
    "usage_engagement",
    "plan_changes"
]

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()

        print(f"{table}: {count:,} rows")

customers: 10,000 rows
subscriptions: 10,113 rows
payments: 109,303 rows
marketing_spend: 180 rows
usage_engagement: 146,190 rows
plan_changes: 2,887 rows


In [38]:
with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT *
            FROM customers
            LIMIT 10
        """)
    )

    customer_df = pd.DataFrame(
        result.fetchall(),
        columns=result.keys()
    )

display(customer_df)

,customer_id,signup_date,country,company_size,acquisition_channel
0,CUST000001,2026-04-02,France,10-Jan,Organic
1,CUST000002,2025-05-23,Brazil,10-Jan,Paid Search
2,CUST000003,2026-06-07,Singapore,10-Jan,Organic
3,CUST000004,2026-02-20,India,51-200,Email
4,CUST000005,2024-03-18,United Kingdom,Nov-50,Organic
5,CUST000006,2026-10-07,India,201-1000,Paid Search
6,CUST000007,2026-04-15,United States,201-1000,Organic
7,CUST000008,2026-04-02,Singapore,10-Jan,Social
8,CUST000009,2024-05-22,Canada,51-200,Organic
9,CUST000010,2025-06-10,Canada,10-Jan,Organic


In [39]:
with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT *
            FROM subscriptions
            LIMIT 10
        """)
    )

    subscription_df = pd.DataFrame(
        result.fetchall(),
        columns=result.keys()
    )

display(subscription_df)

,subscription_id,customer_id,plan,billing_cycle,price,start_date,end_date,status
0,SUB0000001,CUST002394,Pro,annual,790.00,2023-11-01,2024-11-01,cancelled
1,SUB0000002,CUST000744,Pro,monthly,79.00,2023-11-01,2026-10-01,cancelled
2,SUB0000003,CUST007122,Pro,annual,790.00,2023-11-01,2026-10-01,cancelled
3,SUB0000004,CUST000766,Enterprise,monthly,249.00,2023-11-01,None,active
4,SUB0000005,CUST006737,Pro,monthly,79.00,2023-11-01,None,active
5,SUB0000006,CUST007009,Enterprise,monthly,249.00,2023-11-01,2026-10-01,cancelled
6,SUB0000007,CUST009005,Pro,monthly,79.00,2023-11-01,2026-10-01,cancelled
7,SUB0000008,CUST003503,Pro,monthly,79.00,2023-11-01,None,active
8,SUB0000009,CUST007974,Basic,monthly,29.00,2023-11-02,None,active
9,SUB0000010,CUST004315,Pro,monthly,79.00,2023-11-02,None,active


In [40]:
from sqlalchemy import text

tables = [
    "customers",
    "subscriptions",
    "payments",
    "marketing_spend",
    "usage_engagement",
    "plan_changes"
]

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar()

        print(f"{table}: {count:,} rows")

customers: 10,000 rows
subscriptions: 10,113 rows
payments: 109,303 rows
marketing_spend: 180 rows
usage_engagement: 146,190 rows
plan_changes: 2,887 rows
